# Глава 11: Построение Трансформера

> «Мы есть то, что делаем постоянно. Совершенство, следовательно, это не действие, а привычка.»
> — **Аристотель**, Философ

---

## Что вы изучите

- Как складывать блоки трансформера в полноценную языковую модель
- Паттерн конфигурации, упрощающий экспериментирование с моделью
- Что делает голова языкового моделирования (language modeling head) и зачем она нужна
- Связывание весов (weight tying): элегантный трюк, экономящий 38 миллионов параметров
- Важные проверки для валидации модели перед обучением
- Как загружать предобученные веса GPT-2 в вашу архитектуру

---

## Настройка

Сначала установим необходимые пакеты:

In [ ]:
# Установка необходимых пакетов
!pip install -q torch transformers

In [ ]:
# ===== ИМПОРТЫ =====
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from transformers import AutoTokenizer

# Установка устройства
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Используемое устройство: {device}")

## 1. Компоненты из главы 10

Сначала возьмём компоненты, которые мы построили в главе 10: `MultiHeadAttention`, `FeedForward` и `TransformerBlock`.

In [ ]:
# ===== МНОГОГОЛОВОЕ ВНИМАНИЕ (из главы 10) =====
class MultiHeadAttention(nn.Module):
    """
    Эффективное многоголовое внимание (все головы обрабатываются вместе).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model должен делиться на num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Объединённая QKV проекция (в 3 раза эффективнее отдельных!)
        # Почему 3 * d_model? Потому что мы проецируем в Q, K, V одновременно
        # bias=True соответствует GPT-2, поэтому позже мы сможем загрузить его предобученные смещения
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Выходная проекция
        self.out_proj = nn.Linear(d_model, d_model)

        # Dropout на весах внимания
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Шаг 1: Проекция в Q, K, V (всё сразу!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Шаг 2: Разделение на Q, K, V и изменение формы для многоголового =====
        # Изменяем форму на (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # Переставляем в (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Разделяем на Q, K, V: каждый имеет форму (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Шаг 3: Масштабированное скалярное произведение внимания (батчи по головам) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Применяем причинную маску, если она предоставлена
        if mask is not None:
            # Расширяем маску для голов: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Взвешенная сумма значений
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Шаг 4: Конкатенация голов =====
        # Транспонируем в (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)

        # Изменяем форму на (batch, seq, d_model), это конкатенирует головы
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Шаг 5: Финальная проекция =====
        output = self.out_proj(attn_output)

        return output, attn_weights


# Тестируем

In [ ]:
# ===== ПОЛНОСВЯЗНАЯ СЕТЬ (из главы 10) =====
class FeedForward(nn.Module):
    """
    Позиционная полносвязная сеть.
    Применяется к каждой позиции независимо (одинаковые веса для всех позиций).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Args:
            d_model: Размерность модели (768 для GPT-2 small)
            d_ff: Размерность полносвязной сети (обычно 4 * d_model = 3072)
            dropout: Вероятность dropout
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), расширение
        x = F.gelu(x, approximate="tanh")  # Нелинейность (вариант GELU из GPT-2)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), проекция обратно
        return x


# Тестируем

print("FeedForward определён!")

In [ ]:
# ===== БЛОК ТРАНСФОРМЕРА (из главы 10) =====
class TransformerBlock(nn.Module):
    """
    Полный блок трансформера с многоголовым вниманием, полносвязной сетью,
    остаточными соединениями и нормализацией слоя (стиль пре-нормализации, как в GPT-2).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Args:
            d_model: Размерность модели (768)
            num_heads: Количество голов внимания (12)
            d_ff: Размерность полносвязной сети (3072 = 4 * d_model)
            dropout: Вероятность dropout
        """
        super().__init__()

        # Нормализация слоя (перед каждым подслоем)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Многоголовое внимание
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # Полносвязная сеть
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (применяется после каждого подслоя)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Args:
            x: Входные эмбеддинги (batch, seq, d_model)
            mask: Причинная маска (seq, seq) или (1, 1, seq, seq)

        Returns:
            x: Выходные эмбеддинги (batch, seq, d_model)
            attn_weights: Веса внимания (batch, heads, seq, seq)
        """
        # ===== Многоголовое внимание с остаточным соединением =====
        # Пре-нормализация: Нормализуем ПЕРЕД вниманием
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Остаточное соединение

        # ===== Полносвязная сеть с остаточным соединением =====
        # Пре-нормализация: Нормализуем ПЕРЕД полносвязной сетью
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Остаточное соединение

        return x, attn_weights


# Тестируем полный блок трансформера

## 2. Конфигурация модели

Создадим класс данных (dataclass) для конфигурации, объединяющий все гиперпараметры модели.

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Конфигурация для модели MiniGPT."""
    vocab_size: int = 50257      # Размер словаря GPT-2
    max_seq_len: int = 1024      # Максимальная длина контекста
    embed_dim: int = 768         # Размерность эмбеддинга (d_model)
    num_heads: int = 12          # Количество голов внимания
    num_layers: int = 12         # Количество блоков трансформера
    d_ff: int = 3072             # Размерность скрытого слоя полносвязной сети (4 * embed_dim)
    dropout: float = 0.1         # Вероятность дропаута

    def __post_init__(self):
        """Проверка конфигурации после инициализации."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) должен делиться на " \
            f"num_heads ({self.num_heads})"

## 3. Полная модель MiniGPT

Теперь построим полную модель, складывая блоки трансформера и добавляя голову языкового моделирования со связыванием весов.

In [ ]:
class MiniGPT(nn.Module):
    """
    Минимальная языковая модель в стиле GPT.
    Объединяет эмбеддинги, блоки трансформера и голову языкового моделирования.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Эмбеддинги токенов и позиций (глава 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Блоки трансформера (глава 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Финальная нормализация слоя =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Голова языкового моделирования =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Связывание весов =====
        # Общие веса между эмбеддингом и выходной проекцией
        self.lm_head.weight = self.token_embed.weight

        # Инициализация весов
        self._init_weights()

    def _init_weights(self):
        """Инициализация весов небольшими случайными значениями."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Прямой проход через модель.

        Args:
            token_ids: ID входных токенов (batch, seq)
            return_attention: Возвращать ли веса внимания

        Returns:
            logits: Оценки для словаря (batch, seq, vocab_size)
            attention_weights: Необязательный список весов внимания по блокам
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Эмбеддинги =====
        # Эмбеддинги токенов
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Позиционные эмбеддинги
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Объединяем и применяем dropout
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Создание причинной маски =====
        # Нижнетреугольная: каждая позиция может смотреть только на предыдущие позиции
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Проход через блоки трансформера =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Финальная нормализация слоя =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Голова языкового моделирования =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits


    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Генерация токенов, продолжающих input_ids (промпт включён в вывод).

        Args:
            input_ids: ID токенов промпта (batch, seq)
            max_new_tokens: Сколько новых токенов сгенерировать
            temperature: Температура сэмплирования (используется только при do_sample=True)
            do_sample: Сэмплировать из распределения; False = жадный argmax
            eos_token_id: Остановиться раньше, если каждая последовательность сгенерировала этот токен

        Returns:
            ID токенов (batch, seq + новые токены)
        """
        self.eval()

        # Позиционные эмбеддинги покрывают только max_seq_len позиций
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Промпт ({input_ids.shape[1]} токенов) + max_new_tokens "
                f"({max_new_tokens}) превышает контекст модели в {limit} "
                "токенов. Сократите промпт или уменьшите max_new_tokens.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids

In [ ]:
# Быстрый тест с минимальной конфигурацией
tiny_config = GPTConfig(vocab_size=500, max_seq_len=16, embed_dim=64,
                        num_heads=2, num_layers=2, d_ff=256, dropout=0.0)
model = MiniGPT(tiny_config)

# Тестирование прямого прохода
test_tokens = torch.randint(0, tiny_config.vocab_size, (2, 16))
logits = model(test_tokens)

print(f"Форма входа: {test_tokens.shape}")
print(f"Форма выхода: {logits.shape}")
print(f"\nОжидается: (2, 16, {tiny_config.vocab_size})")

## 4. Проверки работоспособности

Проверим, что наша модель правильно собрана, с помощью 5 ключевых тестов. Шестой тест сравнит логиты после загрузки предобученных весов GPT-2.

In [ ]:
# ===== ТЕСТ 1: Проверка формы =====

def test_forward_shapes(config):
    """Проверка формы выходных данных модели."""
    model = MiniGPT(config)
    
    batch_size, seq_len = 2, 16
    token_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))
    
    logits = model(token_ids)
    
    expected_shape = (batch_size, seq_len, config.vocab_size)
    assert logits.shape == expected_shape, \
        f"Ожидалось {expected_shape}, получено {logits.shape}"
    
    print(f"Проверка формы ПРОЙДЕНА: {logits.shape}")

test_forward_shapes(tiny_config)

In [ ]:
# ===== ТЕСТ 2: Количество параметров =====
def count_parameters(model):
    """Подсчёт общего количества обучаемых параметров."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def test_parameter_count(config):
    """Проверка разумности количества параметров."""
    model = MiniGPT(config)
    total = count_parameters(model)

    # У GPT-2 Small ~124M параметров
    print(f"Всего параметров: {total:,}")

    # Точная разбивка (включая члены смещения)
    embed_params = config.vocab_size * config.embed_dim  # Эмбеддинги токенов
    pos_params = config.max_seq_len * config.embed_dim   # Позиционные эмбеддинги
    d = config.embed_dim
    block_params = config.num_layers * (
        4 * d * d + 4 * d +                     # Внимание: QKV + выход, со смещениями
        2 * d * config.d_ff + config.d_ff + d + # Полносвязная сеть: fc1 + fc2, со смещениями
        4 * d                                   # Нормализации слоёв
    )
    ln_f_params = 2 * d                          # Финальная нормализация слоя
    # Примечание: параметры lm_head связаны с token_embed, поэтому отдельно не считаются

    expected = embed_params + pos_params + block_params + ln_f_params
    print(f"Ожидается: {expected:,}")
    assert total == expected, f"Несовпадение количества: {total:,} против {expected:,}"

# Тест с минимальной конфигурацией
print("Минимальная модель:")
test_parameter_count(tiny_config)

# Тест с полной конфигурацией GPT-2
print("\nGPT-2 Small:")
test_parameter_count(GPTConfig())

In [ ]:
# ===== ТЕСТ 3: Причинное маскирование =====

def test_causal_masking(config):
    """Проверка, что модель не видит будущие токены."""
    model = MiniGPT(config)
    model.eval()

    # Одинаковый префикс, разный суффикс
    tokens_a = torch.tensor([[100, 200, 300, 400]])
    tokens_b = torch.tensor([[100, 200, 300, 401]])  # Последний токен отличается

    with torch.no_grad():
        logits_a = model(tokens_a)
        logits_b = model(tokens_b)

    # Позиции 0, 1, 2 должны быть ОДИНАКОВЫМИ
    for pos in range(3):
        assert torch.allclose(logits_a[0, pos], logits_b[0, pos], atol=1e-5), \
            f"Логиты позиции {pos} различаются!"

    # Позиция 3 ДОЛЖНА различаться
    assert not torch.allclose(logits_a[0, 3], logits_b[0, 3], atol=1e-5), \
        "Логиты позиции 3 одинаковы, несмотря на разные входы!"

    print("Причинное маскирование ПРОЙДЕНО!")

test_causal_masking(tiny_config)

In [ ]:
# ===== ТЕСТ 4: Связывание весов =====

def test_weight_tying(config):
    """Проверка, что эмбеддинг и lm_head используют общие веса."""
    model = MiniGPT(config)

    # Должен быть ОДИН И ТОТ ЖЕ тензор
    assert model.lm_head.weight is model.token_embed.weight, \
        "Связывание весов не работает: разные тензоры!"

    # Изменяем один, проверяем, что другой изменился
    with torch.no_grad():
        original = model.token_embed.weight[0, 0].item()
        model.token_embed.weight[0, 0] = 999.0
        
        assert model.lm_head.weight[0, 0].item() == 999.0, \
            "Связывание весов не работает: изменения не распространяются!"
        
        model.token_embed.weight[0, 0] = original

    print("Связывание весов ПРОЙДЕНО!")

test_weight_tying(tiny_config)

In [ ]:
# ===== ТЕСТ 5: Поток градиентов =====

def test_gradient_flow(config):
    """Проверка, что градиенты достигают всех параметров."""
    model = MiniGPT(config)

    # Прямой проход
    tokens = torch.randint(0, config.vocab_size, (1, 8))
    logits = model(tokens)

    # Обратный проход
    loss = logits.sum()
    loss.backward()

    # Проверяем, что у всех параметров есть градиенты
    for name, param in model.named_parameters():
        assert param.grad is not None, f"Нет градиента для {name}"

    print("Поток градиентов ПРОЙДЕН!")

test_gradient_flow(tiny_config)

## 5. Ваш первый прямой проход

Пропустим реальный текст через нашу модель и посмотрим, что она выдаст со случайными весами.

In [ ]:
# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Создание небольшой модели
config = GPTConfig(num_layers=2, embed_dim=256, num_heads=4, d_ff=1024)
model = MiniGPT(config)
model.eval()

# Токенизация промпта
prompt = "The quick brown fox"
token_ids = tokenizer.encode(prompt, return_tensors="pt")

print(f"Промпт: '{prompt}'")
print(f"ID токенов: {token_ids}")
print(f"Токены: {[tokenizer.decode([t]) for t in token_ids[0]]}")

In [ ]:
# Прямой проход
with torch.no_grad():
    logits = model(token_ids)

print(f"Форма логитов: {logits.shape}")

# Получаем топ-5 предсказаний для следующего токена
last_logits = logits[0, -1, :]
top_probs, top_indices = torch.softmax(last_logits, dim=-1).topk(5)

print(f"\nТоп-5 предсказаний после '{prompt}':")
for prob, idx in zip(top_probs, top_indices):
    token = tokenizer.decode([idx])
    print(f"  '{token}': {prob:.4f}")

print("\n(Случайные предсказания, модель имеет случайные веса!)")

## 6. Загрузка предобученных весов

Теперь самое интересное: загрузим настоящие веса GPT-2 в наш MiniGPT!

In [ ]:
def load_gpt2_weights(model, model_name="gpt2"):
    """
    Загрузка предобученных весов GPT-2 в нашу модель MiniGPT.

    Args:
        model: Наш экземпляр MiniGPT
        model_name: Имя модели HuggingFace ("gpt2", "gpt2-medium" и т.д.)

    Returns:
        модель с загруженными весами
    """
    from transformers import GPT2LMHeadModel

    print(f"Загрузка весов из '{model_name}'...")

    # Загружаем модель HuggingFace
    hf_model = GPT2LMHeadModel.from_pretrained(model_name)
    hf_state = hf_model.state_dict()

    # Словарь состояния нашей модели
    our_state = model.state_dict()

    # Копируем эмбеддинги
    our_state['token_embed.weight'].copy_(hf_state['transformer.wte.weight'])
    our_state['pos_embed.weight'].copy_(hf_state['transformer.wpe.weight'])

    # Копируем каждый блок трансформера
    for i in range(model.config.num_layers):
        # Нормализации слоёв
        our_state[f'blocks.{i}.ln1.weight'].copy_(
            hf_state[f'transformer.h.{i}.ln_1.weight'])
        our_state[f'blocks.{i}.ln1.bias'].copy_(
            hf_state[f'transformer.h.{i}.ln_1.bias'])
        our_state[f'blocks.{i}.ln2.weight'].copy_(
            hf_state[f'transformer.h.{i}.ln_2.weight'])
        our_state[f'blocks.{i}.ln2.bias'].copy_(
            hf_state[f'transformer.h.{i}.ln_2.bias'])

        # Внимание (веса нужно транспонировать; смещения копируются напрямую)
        our_state[f'blocks.{i}.attn.qkv_proj.weight'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_attn.weight'].T)
        our_state[f'blocks.{i}.attn.qkv_proj.bias'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_attn.bias'])
        our_state[f'blocks.{i}.attn.out_proj.weight'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_proj.weight'].T)
        our_state[f'blocks.{i}.attn.out_proj.bias'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_proj.bias'])

        # Полносвязная сеть (нужно транспонировать!)
        our_state[f'blocks.{i}.ffn.fc1.weight'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_fc.weight'].T)
        our_state[f'blocks.{i}.ffn.fc1.bias'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_fc.bias'])
        our_state[f'blocks.{i}.ffn.fc2.weight'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_proj.weight'].T)
        our_state[f'blocks.{i}.ffn.fc2.bias'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_proj.bias'])

    # Финальная нормализация слоя
    our_state['ln_f.weight'].copy_(hf_state['transformer.ln_f.weight'])
    our_state['ln_f.bias'].copy_(hf_state['transformer.ln_f.bias'])

    # Примечание: lm_head.weight связан с token_embed.weight, уже скопирован

    print("Веса успешно загружены!")
    return model

In [ ]:
@torch.no_grad()
def generate_simple(model, tokenizer, prompt, max_new_tokens=20):
    """
    Генерация текста жадным декодированием (всегда выбирается самый вероятный токен).

    Args:
        model: Наш MiniGPT с загруженными весами
        tokenizer: Токенизатор GPT-2
        prompt: Начальный текст
        max_new_tokens: Сколько новых токенов сгенерировать

    Returns:
        Строка сгенерированного текста
    """
    device = next(model.parameters()).device
    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # do_sample=False = жадное: argmax на каждом шаге
    output_ids = model.generate(
        token_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Декодируем обратно в текст (промпт включён)
    return tokenizer.decode(output_ids[0])

In [ ]:
# Создаём модель с конфигурацией GPT-2 Small
config = GPTConfig()  # Значения по умолчанию соответствуют GPT-2 Small
model = MiniGPT(config)

# Загружаем предобученные веса
model = load_gpt2_weights(model, "gpt2")
model = model.to(device)

print(f"\nМодель на устройстве: {device}")
print(f"Параметров: {count_parameters(model):,}")

In [ ]:
# ===== ТЕСТ 6: Совпадение логитов с HuggingFace GPT-2 =====
def test_gpt2_parity(model, tokenizer):
    """Проверка, что наша загруженная модель совпадает с логитами HuggingFace GPT-2."""
    from transformers import GPT2LMHeadModel

    hf_model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
    model.eval()

    # Сравниваем на устройстве модели (HF-модель переносится туда же)
    device = next(model.parameters()).device
    hf_model = hf_model.to(device)
    ids = tokenizer.encode("The quick brown fox jumps over",
                           return_tensors="pt").to(device)
    with torch.no_grad():
        our_logits = model(ids)
        hf_logits = hf_model(ids).logits

    # Крошечные различия float допустимы; что-то большее означает потерянный вес
    torch.testing.assert_close(our_logits, hf_logits, atol=1e-4, rtol=1e-4)
    print("Совпадение с GPT-2 ПРОЙДЕНО!")

test_gpt2_parity(model, tokenizer)
# Ожидается: Совпадение с GPT-2 ПРОЙДЕНО!

In [ ]:
# Генерируем текст!
prompt = "The quick brown fox"
generated = generate_simple(model, tokenizer, prompt, max_new_tokens=30)

print(f"Промпт: '{prompt}'")
print(f"Сгенерировано: '{generated}'")

In [ ]:
# Попробуем больше промптов!
prompts = [
    "Artificial intelligence will",
    "Once upon a time",
    "The capital of France is",
    "def fibonacci(n):"
]

for prompt in prompts:
    generated = generate_simple(model, tokenizer, prompt, max_new_tokens=20)
    print(f"'{prompt}' -> {generated}")
    print()

## 7. Сравнение случайных и предобученных весов

Наглядно сравним случайные веса с предобученными весами.

In [ ]:
# Модель со случайными весами
model_random = MiniGPT(GPTConfig())
model_random = model_random.to(device)

prompt = "Artificial intelligence will"

print("=" * 50)
print("СЛУЧАЙНЫЕ ВЕСА:")
print(generate_simple(model_random, tokenizer, prompt, max_new_tokens=20))
print()
print("ПРЕДОБУЧЕННЫЕ ВЕСА:")
print(generate_simple(model, tokenizer, prompt, max_new_tokens=20))
print("=" * 50)

print("\nОдинаковая архитектура. Один и тот же код. Обучение делает всю разницу!")

## 8. Упражнения

### Упражнение 1: Минимальная модель

Постройте и протестируйте минимальную модель с определёнными параметрами.

In [ ]:
# ВАШ КОД ЗДЕСЬ
# Постройте минимальную MiniGPT со следующими параметрами:
# - vocab_size=500
# - max_seq_len=16
# - embed_dim=64
# - num_heads=2
# - num_layers=2
# - d_ff=256

# 1. Создайте конфигурацию
# 2. Постройте модель
# 3. Выполните прямой проход на случайных токенах
# 4. Проверьте форму выхода
# 5. Подсчитайте параметры

### Упражнение 2: Исследование температуры

Модифицируйте генерацию для использования масштабирования температурой.

In [ ]:
# ВАШ КОД ЗДЕСЬ
# Модифицируйте generate_simple для принятия параметра температуры:
# next_logits = next_logits / temperature
# 
# Попробуйте температуры: 0.5, 1.0, 1.5
# Как изменяется выход?

### Упражнение 3: Визуализация внимания

Визуализируйте паттерны внимания в предобученной модели.

In [ ]:
# ВАШ КОД ЗДЕСЬ
# 1. Запустите модель с return_attention=True
# 2. Получите веса внимания из последнего блока
# 3. Постройте тепловую карту для головы 0
# Подсказка: используйте matplotlib.pyplot.imshow()

### Предпросмотр групп параметров (используется в главе 12)

Глава 12 применяет затухание весов только к тем параметрам, которым оно полезно. Этот помощник реализует разделение: матрицы (2D+) получают затухание, смещения и векторы LayerNorm — нет.


In [ ]:
def get_parameter_groups(model, weight_decay=0.1):
    """
    Разделяет параметры, которым нужно/не нужно затухание весов.

    Затухание весов — регуляризация, поощряющая меньшие веса.
    Мы не применяем его к:
    - параметрам LayerNorm (они должны свободно масштабироваться)
    - смещениям (они не склонны к переобучению)
    - эмбеддинги — 2D-матрицы, поэтому здесь затухание ПОЛУЧАЮТ (правило ndim)
    """
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        # Матрицы (2D+) получают затухание весов; векторы (смещения, масштабы LayerNorm) — нет
        if param.ndim < 2:
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ]

# Глава 12 строит оптимизатор с этим помощником:
# AdamW(get_parameter_groups(model, weight_decay=0.01), lr=learning_rate)
print("get_parameter_groups определён!")


## Резюме

**Что мы построили:**

1. **GPTConfig**: Чистый паттерн конфигурации
2. **MiniGPT**: Полная языковая модель со связыванием весов
3. **Проверки работоспособности**: 6 тестов, включая совпадение логитов с HuggingFace GPT-2
4. **Загрузка весов**: Перенос весов GPT-2
5. **Генерация текста**: Жадное декодирование

**Ключевые концепции:**

- `nn.ModuleList` для стекирования слоёв
- Связывание весов экономит 38 миллионов параметров
- Голова языкового моделирования: `(batch, seq, embed_dim)` → `(batch, seq, vocab_size)`
- `model.train()` против `model.eval()`
- Ловушка связывания весов: `'lm_head.weight'` ДЕЙСТВИТЕЛЬНО присутствует в `state_dict().keys()`; связанные веса делят хранилище, `torch.save` пишет его один раз, и файл остаётся маленьким

**Далее:** Глава 12 научит вас тренировать эту модель с нуля!